In [0]:
%run "../notebooks/helper_functions"

#### Imports and setup

This cell imports helper functions and creates a silver_run_id for the current run

In [0]:
from pyspark.sql.window import Window
import uuid

In [0]:
silver_run_id = str(uuid.uuid4())
print(f"Current silver run id: {silver_run_id}")

#### Orders incremental processing
This cell processes **orders** from Bronze to Silver.
It does the following
- reads only new Bronze order rows
- cleans values like order_status and order_amount
- keeps only the latest version per order_id
- validates business rules
- sends bad rows to quarantine
- merges good rows into orders_transformed

In [0]:
df_raw = spark.sql("select * from novacart_catalog.bronze_schema.orders_raw").limit(50)
display(df_raw)

In [0]:
# Read only the Bronze order rows that Silver has not processed
orders_inc, last_orders_ingested_at = get_incremental_bronze("novacart_catalog.bronze_schema.orders_raw", "orders")

# Count the incremental order rows entering Silver in this run
orders_inc_count = orders_inc.count()
print(f"orders rows_to_process_in_silver: {orders_inc_count}")

# Only run Silver order cleaning and validation when there are new Bronze order rows
if orders_inc_count > 0:
  # Create a window that keeps the latest order for each order_id
  order_window = Window.partitionBy("order_id").orderBy(
    F.col("updated_at").cast("timestamp").desc(),
    F.col("bronze_ingested_at").desc()
  )
  
  # Start the silver order-cleaning pipeline. This block standardizes and deduplicates raw order records.
  orders_cleaned = (
    orders_inc
    # Standardize order_status to uppercase so valudes surch as shipped and SHIPPED become consistent
    .withColumn("orders_status", F.upper(F.trim(F.col("order_status"))))
    .withColumn("order_status", F.when(F.col("order_status") == "", F.lit(None)).otherwise(F.col("order_status")))
    # Remove formatting characters from order_amount so it can be cast to a numeric type
    .withColumn("order_amount", F.regexp_replace(F.col("order_amount"), r"[$, ]", ""))
    .withColumn("order_amount", F.when(F.trim(F.col("order_amount")).isin("N/A", "NULL", "??", ""), None).otherwise(F.col("order_amount")))
    .withColumn("order_amount", F.col("order_amount").cast("double"))
    .withColumn("created_at", F.to_timestamp("created_at"))
    .withColumn("updated_at", F.to_timestamp("updated_at"))
    # Assign a row number inside each business key so we can keep only the latest version of that record
    .withColumn("row_rank", F.row_number().over(order_window))
    # Keep only the latest record for each business key.
    .filter(F.col("row_rank") == 1)
    .drop("row_rank")
    .withColumn("silver_run_id", F.lit(silver_run_id))
  )

  # Merge the cleaned or validated Silver dataset into its Delta target table
  upsert_to_silver(
    orders_cleaned,
    "novacart_catalog.silver_schema.orders_cleaned",
    ["order_id"]
  )

  # Apply silver data-quality rules to the cleaned order records
  orders_validated = (
    orders_cleaned
    .withColumn(
      "to_be_verified_by_orders_team",
      F.when(F.col("customer_id").isNull(), "verify_customer_id")
      .when(F.col("product_id").isNull(), "verify_product_id")
      .when(F.col("order_status").isNull() | (F.trim(F.col("order_status")) == ""), "verify_order_status")
      .when(F.col("order_amount").isNull() | (F.trim(F.col("order_amount").cast("bigint")) <= 0), "verify_order_amount")
      .otherwise("No issues")     
    )
  .withColumn(
    "check_order_amount",
    F.when(F.col("order_amount").isNull() | (F.col("order_amount") <= 0), F.lit(True))
    .otherwise(F.lit(False))
  )
  .withColumn("order_date", F.to_date("created_at"))
  .withColumn("order_year", F.year("created_at"))
  .withColumn("order_month", F.month("created_at"))
  .withColumn("order_day", F.dayofmonth("created_at"))
  .withColumn("order_dow", F.date_format("created_at", "E"))
  )

  # Keep only valid order rows for the transformed silver table
  orders_good_df = orders_validated.filter(F.col("to_be_verified_by_orders_team") == "No issues")
  # Send invalid order rows to the quarantine dataset for manual review
  orders_bad = (
    orders_validated
    .filter(F.col("to_be_verified_by_orders_team") != "No issues")
    .withColumn("quarantine_ts", F.current_timestamp())
  )


  # Merge the cleaned or validated silver dataset into its Delta target table.
  upsert_to_silver(
    orders_good_df,
    "novacart_catalog.silver_schema.orders_transformed",
    ["order_id"]
  )

  # Append bad order rows to the quarantine table instead of losing them
  orders_bad.write.format("delta").mode("append").saveAsTable("novacart_catalog.silver_schema.orders_quarantine")

  mx_ingested = orders_inc.agg(F.max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]

  mx_run_id = (
    orders_inc.filter(F.col("bronze_ingested_at") == F.lit(mx_ingested))
    .agg(F.max("bronze_run_id").alias("mx"))
    .collect()[0]["mx"]
  )

  orders_good_df_count = orders_good_df.count()

  upsert_silver_control("orders", mx_run_id, mx_ingested, orders_good_df_count, silver_run_id)

else:
  print("No new orders Bronze rows for Silver")
  
  upsert_silver_control(
    "orders",
    None,
    last_orders_ingested_at,
    orders_inc_count,
    silver_run_id
  )

#### Products incremental processing
This cell processes **products** from Bronze to Silver
It handles 
- product name cleanup
- category standardization
- price cleanup and numeric conversion
- latest-record selection per product_id
- data quality validation
- quarantine for bad rows
- merge into Silver current-state tables

In [0]:
# Step 5 - Products incremental processing
# Read only the Bronze product rows that Silver have not processed yet
products_inc, last_products_ingested_at = get_incremental_bronze("novacart_catalog.bronze_schema.products_raw", "products")

# Count the incremental product rows entering Silver in this run.
products_inc_count = products_inc.count()
print(f"products row_to_process_in_silver = {products_inc_count}")

if products_inc_count() > 0:
  # Create a window that keeps the latest product record for each product_id
  prodcut_window = Window.partitionBy("product_id").orderBy(
    F.col("updated_at").cast("timestamp").desc(),
    F.col("bronze_ingested_at").desc()
  )

#  Start the silver product-cleaning pipeline. This block standardizes and deduplicates raw product records.
products_cleaned = (
  products_inc
  # Standardize product_name by trimming spaces and converting text to uppercase.
  .withColumn("product_name", F.upper(F.trim(F.col("product_name"))))
  .withColumn("product_name", F.when(F.col("product_name") == "", F.lit(None)).otherwise(F.col("product_name")))
  .withColumn(
    "category",
    F.when(F.upper(F.trim(F.col("category"))).contains("ELECTRONICS"), "ELECTRONICS")
  )
  .withColumn()
  )